In [1]:
import numpy as np
import time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
import warnings
warnings.filterwarnings('ignore',category=FutureWarning)


In [2]:
print("Downloading Fashion MNIST dataset(this may take 30-60 seconds)...")
fashion_mnist=fetch_openml('Fashion-MNIST', version=1, as_frame=False)
x=fashion_mnist.data
y=fashion_mnist.target.astype(int)
print(f"Total dataset size: {x.shape[0]} images,each with {x.shape[1]} pixels.")

Total dataset size: 70000 images,each with 784 pixels.


In [3]:
x_subset, _, y_subset, _=train_test_split(
    x,y,
    train_size=12000,
    stratify=y,
    random_state=42
)

x_train,x_test,y_train,y_test=train_test_split(
    x_subset,y_subset,
    test_size=2000,
    stratify=y_subset,
    random_state=42
)

print(f"Training images: {x_train.shape[0]}")
print(f"Training images: {x_test.shape[0]}")

Training images: 10000
Training images: 2000


In [4]:
print(f"Before scaling ->Min: {x_train.min()},Max: {x_train.max()}")
x_train=x_train/ 255.0
x_test=x_test/ 255.0
print(f"After scaling -> Min: {x_train.min()},Max: {x_train.max()}")

Before scaling ->Min: 0,Max: 255
After scaling -> Min: 0.0,Max: 1.0


In [6]:
k_values=[ 1,3,5,7,9,15]
results={}
print(f"{'K value':<8} |{'Accuracy':<10} | {'Prediction Time (seconds)':<25}")
print("_" * 50)

for k in k_values:
    knm=KNeighborsClassifier(n_neighbors=k, metric='euclidean',n_jobs=-1)
    knm.fit(x_train,y_train)
    start_time = time.time()
    y_pred=knm.predict(x_test)
    elapsed_time=time.time() - start_time
    acc=accuracy_score(y_test,y_pred)

    results[k]={
    "accuracy": acc,
    "time": elapsed_time,
    "predictions": y_pred
    }
    print(f"{k:<8} | {acc* 100:<9.2f}% | {elapsed_time:<25.2f}")   
    

K value  |Accuracy   | Prediction Time (seconds)
__________________________________________________
1        | 79.30    % | 0.14                     
3        | 80.90    % | 0.14                     
5        | 81.00    % | 0.13                     
7        | 81.10    % | 0.14                     
9        | 80.95    % | 0.14                     
15       | 80.25    % | 0.14                     


In [7]:
class_names=[
    "T-shirt/top","Trouser","Pullover", "Dress", "Coat",
    "Sandal","Shirt","Sneaker","Bag", "Ankle boot"
]
best_k=max(results,key=lambda k: results[k]["accuracy"])
print(f"Best K is: {best_k} with {results[best_k]['accuracy'] * 100:.2f}% accuracy\n")
print("Per-Class Classification Report:")
print(classification_report(y_test,results[best_k]["predictions"],target_names=class_names))

Best K is: 7 with 81.10% accuracy

Per-Class Classification Report:
              precision    recall  f1-score   support

 T-shirt/top       0.73      0.83      0.78       200
     Trouser       0.97      0.94      0.96       200
    Pullover       0.67      0.73      0.70       200
       Dress       0.85      0.83      0.84       200
        Coat       0.74      0.69      0.72       200
      Sandal       1.00      0.76      0.86       200
       Shirt       0.58      0.55      0.56       200
     Sneaker       0.81      0.92      0.86       200
         Bag       0.97      0.94      0.95       200
  Ankle boot       0.85      0.94      0.89       200

    accuracy                           0.81      2000
   macro avg       0.82      0.81      0.81      2000
weighted avg       0.82      0.81      0.81      2000

